# Cell Reprogramming

> Author: Ruowen Xiao
>
> Created on: 2026-10-15

## 1. Introduction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass
from scipy.integrate import solve_ivp

## 2. Implementation

### 2.1. Parameters Preparation

In [3]:
@dataclass
class ModelParams:
    N_over: float
    O_over: float
    T_over: float

    LIF: float

    p_N: float
    p_O: float
    p_T: float

    n: int

    K_O: float
    K_d: float
    K_NT: float

In [ ]:
params_0 = ModelParams(
    N_over=0,
    O_over=0,
    T_over=0.05,
    LIF=0,

    p_N=1,
    p_O=1,
    p_T=1,
    n=2,
    K_d=0.1,
    K_O=0.3,
    K_NT=0.2
)
params_0

ModelParams(N_over=0, O_over=0, T_over=0.05, LIF=0, p_N=1, p_O=1, p_T=1, n=2, K_O=0.3, K_d=0.1, K_NT=0.2)

In [ ]:
params_N = ModelParams(
    N_over=0.3,
    O_over=0,
    T_over=0.05,
    LIF=0.06,

    p_N=1,
    p_O=1,
    p_T=1,
    n=2,
    K_d=0.1,
    K_O=0.3,
    K_NT=0.2
)
params_N

ModelParams(N_over=0.3, O_over=0, T_over=0.05, LIF=0.06, p_N=1, p_O=1, p_T=1, n=2, K_O=0.3, K_d=0.1, K_NT=0.2)

In [8]:
params_O = ModelParams(
    N_over=0,
    O_over=0.3,
    T_over=0.05,
    LIF=0.06,

    p_N=1,
    p_O=1,
    p_T=1,
    n=2,
    K_d=0.1,
    K_O=0.3,
    K_NT=0.2
)
params_O

ModelParams(N_over=0, O_over=0.3, T_over=0.05, LIF=0.06, p_N=1, p_O=1, p_T=1, n=2, K_O=0.3, K_d=0.1, K_NT=0.2)

In [9]:
params_T = ModelParams(
    N_over=0,
    O_over=0,
    T_over=0.3,
    LIF=0.06,

    p_N=1,
    p_O=1,
    p_T=1,
    n=2,
    K_d=0.1,
    K_O=0.3,
    K_NT=0.2
)
params_T

ModelParams(N_over=0, O_over=0, T_over=0.3, LIF=0.06, p_N=1, p_O=1, p_T=1, n=2, K_O=0.3, K_d=0.1, K_NT=0.2)

### 2.2 Modeling

Expression of the complex $[N \vert T]$:

TODO: write equations

In [ ]:
def NT(N_total, T_total, K_d):
    a = 1
    b = - (N_total + T_total + K_d)
    c = N_total * T_total
    return (- b - np.sqrt(b**2 - 4 * a * c)) / (2 * a)

Activator Hill equation:

$$A[T] = \frac{[T]^n}{K_A^n + [T]^n}$$

In [ ]:
def hill(activator, K, n):
    temp_term = (activator / K)**n
    return temp_term / (1 + temp_term)

system of ODEs:

$$\begin{cases}
\displaystyle \frac{\partial [N_{total}]}{\partial t} = N_{over} + LIF &+ p_N \cdot A([O_{total}], [N_{total}]) &- [N_{total}] \\
\displaystyle \frac{\partial [O_{total}]}{\partial t} = O_{over} + LIF &+ p_O \cdot A([O_{total}], [O_{total}]) \cdot A([N \vert T], [O_{total}]) &- [O_{total}] \\
\displaystyle \frac{\partial [T_{total}]}{\partial t} = T_{over} &+ p_T \cdot A([O_{total}], [T_{total}]) \cdot A([N \vert T], [T_{total}]) &- [T_{total}] \\
\end{cases}$$

In [ ]:
def f(t, Y, p: ModelParams):
    y_N, y_O, y_T = Y

    dyNdt = p.N_over + p.LIF + p.p_N * hill(y_O, p.K_O, 1) - y_N
    dyOdt = p.O_over + p.LIF + p.p_O * hill(y_O, p.K_O, 1) * hill(NT(y_N, y_T, p.K_d), p.K_NT, p.n) - y_O
    dyTdt = p.T_over + p.p_N * hill(y_O, p.K_O, 1) * hill(NT(y_N, y_T, p.K_d), p.K_NT, p.n) - y_T

    return [dyNdt, dyOdt, dyTdt]

### 2.3 Simulation